# Promo Finder: run on Colab (free)

1. In the left sidebar click the key icon (**Secrets**), add a secret named `GEMINI_API_KEY`, paste your key, switch on *Notebook access*.
2. Run the cells in order. You will upload two files: `promo_finder.zip` and the screenplay pack zip.

In [ ]:
from google.colab import files, userdata
import os, zipfile, glob
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('key loaded:', bool(os.environ['GEMINI_API_KEY']))

In [ ]:
# upload promo_finder.zip (the code)
up = files.upload()
for f in up: zipfile.ZipFile(f).extractall('.')
%cd /content/promo_finder
!ls

In [ ]:
# upload the screenplay pack zip (kept private, do not share or publish it)
up = files.upload()
os.makedirs('/content/pack', exist_ok=True)
def unpack(path):
    z = zipfile.ZipFile(path)
    for n in z.namelist():
        if n.endswith('.zip'):
            z.extract(n, '/content/tmp'); unpack('/content/tmp/'+n)
    z.extractall('/content/pack')
for f in up: unpack(f)
print(len(glob.glob('/content/pack/**/episode-*.txt', recursive=True)), 'episodes found')
PACK = os.path.dirname(glob.glob('/content/pack/**/episode-001*.txt', recursive=True)[0])

In [ ]:
# optional quick test on 3 episodes first
!python -m promo_finder --pack "$PACK" --out /content/outputs_test --episodes 1,2,3 --top 3

In [ ]:
# full run, all 50 episodes (about 10 minutes on the free tier)
# to swap the model: add  --provider anthropic --model <id>  or  --model gemini-2.5-pro
!python -m promo_finder --pack "$PACK" --out /content/outputs --top 15

In [ ]:
print(open('/content/outputs/promos_ranked.md').read()[:4000])

In [ ]:
# download the raw outputs folder
!zip -qr outputs.zip /content/outputs
files.download('outputs.zip')